# 基礎編15
このノートブックでは、トランザクションログをコントラクト内から参照する例を示します。

- 使用するスマートコントラクトAPI:  
queryLastTransaction, queryTransaction


## スマートコントラクトの処理の概要
- トランザクションログを検索して、同じcallerの前回の呼び出しのtxno（トランザクション番号）を取得する。
- txnoからトランザクションの情報を取得する。
- 前回の呼び出しとの整合性が無い場合にはエラーとする。
  （前回の引数bの値と、今回の引数aの値が等しいことが条件）

## 準備

実行の準備を行います。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

あらかじめ用意してあるユーザ／ウォレットを読み込みます。


In [2]:
var users = [];
for(var i=0; i<=1; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu


## スマートコントラクトのデプロイ

スマートコントラクトを便宜的にfunctionとして記述します。

In [3]:
function basic15(a, b){
    var uid = getCallerId();
    if(uid === 'anonymous'){
        throw "anonymous access not allowed";
    }
    var prev_b = '';
    var prev_count = 0;
    var txno = queryLastTransaction(getContractId(), { caller: uid });
    if(txno){
        var tx = queryTransaction(txno);
        var prev_b = tx.args.b;
        var prev_count = tx.value.count;
    }
    if (a !== prev_b) throw "unexpected value of a == " + a + ", expected value: " + prev_b;
    return { count: prev_count+1 };
}

上記のスマートコントラクトをデプロイします。

In [4]:
var cid = await deploySmartContract({ a: 'string', b: 'string' }, basic15);
console.log(cid);

c047135398


デプロイしたスマートコントラクトの呼び出しを、上記user0～user1 に許可します。

In [5]:
await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add accessible_to', value: [users[0].id, users[1].id] });

{
  txno: 701939,
  txid: 'x3ViYyPZxzZvsaXP2UjfJgzYnCfx3uV8ZSLVwkRVskeFq',
  status: 'ok',
  value: null
}

## スマートコントラクトの実行

最初の呼び出しを行います。

In [6]:
var resp = await rpc.call(users[0].wallet, cid, { a: '', b: 'xxx' });
console.log(resp);

{
  txno: 701940,
  txid: 'xxwguzk77jUy7vHtMavVkBmSV62yyVmbuJrsprNYiwBGt',
  status: 'ok',
  value: { count: 1 }
}


２回目の呼び出しを１回目の呼び出しと整合性がある引数で呼び出します。

In [7]:
var resp = await rpc.call(users[0].wallet, cid, { a: 'xxx', b: 'yyy' });
console.log(resp);

{
  txno: 701941,
  txid: 'xBatqrMWMmFtf9gG6QGdjngJboh9hisRHgJRMC6JER73EB',
  status: 'ok',
  value: { count: 2 }
}


３回目の呼び出しを２回目の呼び出しと整合性が*無い*引数で呼び出します。  
整合性が無いため、エラーとなります。

In [8]:
var resp = await rpc.call(users[0].wallet, cid, { a: 'zzz', b: 'www' });
console.log(resp);

{
  txno: 701942,
  txid: 'xBEMWPtZZFBfwStYXZzwxgpdWGbZvcGcUwbDmUGMVXN5r',
  status: 'thrown',
  value: 'unexpected value of a == zzz, expected value: yyy'
}


今度は、２回目の呼び出しと整合性がある引数で呼び出します。  
ポイント：エラーとなったトランザクションはqueryLastTransactionで無視されます。

In [9]:
var resp = await rpc.call(users[0].wallet, cid, { a: 'yyy', b: 'zzz' });
console.log(resp);

{
  txno: 701943,
  txid: 'xEkKThW4q6KdfSVxzXvmJDNQEoGQhHwdEmpcPd7e4Ljhy',
  status: 'ok',
  value: { count: 3 }
}


別のユーザ1で同様の呼び出しを行います。  
ポイント：callerで絞り込んでqueryLastTransactionを呼び出しているため、別のユーザとは交わりません。

In [10]:
var resp = await rpc.call(users[1].wallet, cid, { a: '', b: '123' });
console.log(resp);

{
  txno: 701944,
  txid: 'xyDHnqjsBegWG9Vgcnyi6rqQW428cyUrHd2ch42F4sBADB',
  status: 'ok',
  value: { count: 1 }
}


In [11]:
var resp = await rpc.call(users[1].wallet, cid, { a: '123', b: '456' });
console.log(resp);

{
  txno: 701945,
  txid: 'xd6RXwX2Z3k5LKMqyLxtWtgpy5td2cVb9SNpvntxyHK55',
  status: 'ok',
  value: { count: 2 }
}


ふたたび、ユーザ0で呼び出します。  
やはり、別のユーザ1とは交わりません。

In [12]:
var resp = await rpc.call(users[0].wallet, cid, { a: 'zzz', b: 'www' });
console.log(resp);

{
  txno: 701946,
  txid: 'xYe3eAw4XRvjywznhEVJb85UM43RCYEDZ9cKRze5rSzB5',
  status: 'ok',
  value: { count: 4 }
}
